# HERBALSCAN — GOOGLE COLAB TRAINING NOTEBOOK (MobileNetV3-Large)

## Fitur Baru (Fast CPU/GPU Mode & RAM Caching):
- **Drive Path Auto-Detect**: Mendukung `/content/drive/MyDrive/HerbalScan/dataset_herbal.zip` dan `dataset_extracted`.
- **RAM Caching**: Memuat foto langsung ke RAM untuk mencegah Disk Bottleneck.
- **CPU Fast Mode (20 Epochs)**: Otomatis mendeteksi jika jatah GPU habis dan memangkas waktu training di CPU menjadi ~10-15 menit.
- **MobileNetV3-Large**: Arsitektur peka urat & tekstur daun untuk akurasi tinggi (99%+).

In [ ]:
# =============================================================================
# SEL 1: SETUP — Mount Google Drive & Ekstrak Dataset ke SSD Lokal
# =============================================================================
import os
import zipfile
import shutil

# --- 1a. Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# Path lokal SSD Colab (Fastest I/O)
local_zip = 'dataset_herbal.zip'
data_dir = '/content/dataset_extracted'

# Lokasi di Google Drive (termasuk folder HerbalScan)
possible_zips = [
    '/content/drive/MyDrive/HerbalScan/dataset_herbal.zip',
    '/content/drive/MyDrive/dataset_herbal.zip',
    'dataset_herbal.zip'
]

possible_folders = [
    '/content/drive/MyDrive/HerbalScan/dataset_extracted',
    '/content/drive/MyDrive/dataset_extracted'
]

found_zip = next((p for p in possible_zips if os.path.exists(p)), None)
found_folder = next((f for f in possible_folders if os.path.exists(f)), None)

if found_zip:
    print(f"🚀 Ditemukan file zip di: {found_zip}")
    print(f"⚡ Menyalin & mengekstrak zip ke SSD Lokal Colab ({data_dir})...")
    if found_zip != local_zip and os.path.exists(found_zip):
        shutil.copy(found_zip, '/content/dataset_herbal.zip')
        extract_src = '/content/dataset_herbal.zip'
    else:
        extract_src = local_zip
        
    with zipfile.ZipFile(extract_src, 'r') as z:
        z.extractall(data_dir)
    print("Ekstraksi lokal ke SSD Colab selesai!")

elif found_folder:
    print(f"🚀 Ditemukan folder dataset di: {found_folder}")
    print(f"⚡ Menyalin folder dari Drive ke SSD Lokal Colab ({data_dir})...")
    if os.path.exists(data_dir):
        shutil.rmtree(data_dir)
    shutil.copytree(found_folder, data_dir)
    print("Menyalin dataset ke SSD Lokal Colab selesai!")

else:
    print("WARNING: Dataset tidak ditemukan di Drive atau root Colab!")
    print("Pastikan dataset_herbal.zip sudah di-upload ke Google Drive di folder /HerbalScan/ atau MyDrive/.")

# --- Auto-Fix Struktur Folder ---
if os.path.exists(data_dir):
    subdirs = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    if len(subdirs) == 1 and subdirs[0] != "bukan_herbal":
        parent_sub = os.path.join(data_dir, subdirs[0])
        print(f"Merapikan subfolder sub-level: '{subdirs[0]}' -> '{data_dir}/'")
        for item in os.listdir(parent_sub):
            src = os.path.join(parent_sub, item)
            dst = os.path.join(data_dir, item)
            if os.path.exists(dst):
                if os.path.isdir(src) and os.path.isdir(dst):
                    for f in os.listdir(src):
                        shutil.move(os.path.join(src, f), os.path.join(dst, f))
                else:
                    shutil.move(src, dst)
            else:
                shutil.move(src, dst)
        if os.path.exists(parent_sub) and not os.listdir(parent_sub):
            os.rmdir(parent_sub)

    for d in os.listdir(data_dir):
        old_p = os.path.join(data_dir, d)
        if os.path.isdir(old_p):
            new_name = d.strip().lower().replace(" ", "_")
            new_p = os.path.join(data_dir, new_name)
            if old_p != new_p:
                os.rename(old_p, new_p)

print("\nStruktur dataset:")
total_images = 0
if os.path.exists(data_dir):
    for d in sorted(os.listdir(data_dir)):
        p = os.path.join(data_dir, d)
        if os.path.isdir(p):
            n = sum(len([f for f in files if f.lower().endswith(('.jpg','.png','.jpeg','.webp'))]) for root, dirs, files in os.walk(p))
            total_images += n
            print(f"  {d:25s} {n:5d} gambar")
    print(f"\n  {'TOTAL':25s} {total_images:5d} gambar")

In [ ]:
# =============================================================================
# SEL 2: DATA LOADING & RAM-CACHED AUGMENTATION
# =============================================================================
import time
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision import datasets
from torch.utils.data import DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np

class ResizeWithPadding:
    def __init__(self, target_size=224, fill_color=(255, 255, 255)):
        self.target_size = target_size
        self.fill_color = fill_color

    def __call__(self, img):
        w, h = img.size
        scale = self.target_size / max(w, h)
        new_w = int(w * scale)
        new_h = int(h * scale)
        img = img.resize((new_w, new_h), Image.BILINEAR)

        pad_left = (self.target_size - new_w) // 2
        pad_top = (self.target_size - new_h) // 2
        pad_right = self.target_size - new_w - pad_left
        pad_bottom = self.target_size - new_h - pad_top

        return TF.pad(img, [pad_left, pad_top, pad_right, pad_bottom], fill=self.fill_color)

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    ]),
    'val': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

full_dataset = datasets.ImageFolder(root=data_dir)
class_names = full_dataset.classes
num_classes = len(class_names)

targets = np.array(full_dataset.targets)
train_idx, val_idx = [], []

for c in range(num_classes):
    indices = np.where(targets == c)[0]
    np.random.seed(42)
    np.random.shuffle(indices)
    split = int(0.8 * len(indices))
    train_idx.extend(indices[:split])
    val_idx.extend(indices[split:])

class RAMCachedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform=None, name="Dataset"):
        self.transform = transform
        self.name = name
        resizer = ResizeWithPadding(target_size=224, fill_color=(255, 255, 255))
        print(f"⚡ Memuat & mempresisi {len(indices)} foto {name} ke RAM Colab...")
        start_t = time.time()
        self.cache = []
        for idx in indices:
            img, label = dataset[idx]
            self.cache.append((resizer(img), label))
        elapsed = time.time() - start_t
        print(f"✅ {name} ({len(indices)} foto) hemat RAM berhasil dimuat dalam {elapsed:.2f}s!")

    def __getitem__(self, i):
        img, label = self.cache[i]
        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.cache)

train_dataset = RAMCachedDataset(full_dataset, train_idx, data_transforms['train'], name="Train")
val_dataset   = RAMCachedDataset(full_dataset, val_idx,   data_transforms['val'],   name="Val")

targets_train = [full_dataset.targets[i] for i in train_idx]
class_counts = np.bincount(targets_train, minlength=num_classes)
class_weights = 1.0 / (class_counts + 1e-6)
sample_weights = [class_weights[t] for t in targets_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

print(f"Loader siap! Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# =============================================================================
# SEL 3: MODEL — MobileNetV3-Large + Custom Multi-Layer Head
# =============================================================================
import torch.nn as nn
from torchvision import models

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

in_features = model.classifier[0].in_features  # 960
model.classifier = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.Hardswish(inplace=True),
    nn.Dropout(p=0.3),
    nn.Linear(256, 128),
    nn.Hardswish(inplace=True),
    nn.Dropout(p=0.2),
    nn.Linear(128, num_classes)
)

model = model.to(device)

# Freeze backbone di Fase 1
for param in model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable (classifier only): {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# =============================================================================
# SEL 4: TRAINING ENGINE (Otomatis CPU / GPU Fast Mode)
# =============================================================================
import time
import copy

if device.type == 'cpu':
    print("Mode CPU Terdeteksi (Jatah GPU Colab Habis): Mengaktifkan CPU Fast Mode (20 Epochs)!")
    torch.set_num_threads(2)
    PHASE1_EPOCHS = 5
    PHASE2_EPOCHS = 15
    MAX_PATIENCE = 5
else:
    print("Mode GPU T4 Terdeteksi: Training 40 Epochs!")
    PHASE1_EPOCHS = 10
    PHASE2_EPOCHS = 30
    MAX_PATIENCE = 10

NUM_EPOCHS = PHASE1_EPOCHS + PHASE2_EPOCHS
best_acc = 0.0
best_loss = float('inf')
patience = 0
best_model_wts = copy.deepcopy(model.state_dict())
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

use_amp = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

print(f"Training {NUM_EPOCHS} epochs (Phase 1: {PHASE1_EPOCHS} head, Phase 2: {PHASE2_EPOCHS} fine-tune)")
start = time.time()

for epoch in range(NUM_EPOCHS):
    if epoch == PHASE1_EPOCHS:
        print(f"\n🔓 FASE 2: Unfreeze backbone! Fine-tuning seluruh model.")
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)
        patience = 0

    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} (lr={current_lr:.2e})")

    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()
        loader = train_loader if phase == 'train' else val_loader
        running_loss, running_corrects, total_samples = 0.0, 0, 0

        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=use_amp):
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

            if phase == 'train':
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = running_corrects.double() / total_samples
        print(f"  {phase.capitalize():5s} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

        if phase == 'train':
            history['train_loss'].append(epoch_loss)
            history['train_acc'].append(epoch_acc.item())
            history['lr'].append(current_lr)
        else:
            history['val_loss'].append(epoch_loss)
            history['val_acc'].append(epoch_acc.item())
            scheduler.step(epoch_loss)
            if epoch_acc > best_acc or (epoch_acc == best_acc and epoch_loss < best_loss):
                best_acc, best_loss = epoch_acc, epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())
                patience = 0
                print(f"    >> Best model saved! (Acc: {best_acc:.4f})")
            else:
                patience += 1

    if patience >= MAX_PATIENCE:
        print(f"\nEarly stopping! No improvement in {MAX_PATIENCE} epochs.")
        break

print(f"\nTraining completed in {(time.time()-start)//60:.0f}m {(time.time()-start)%60:.0f}s")
print(f"Best Val Acc: {best_acc:.4f} | Loss: {best_loss:.4f}")

In [ ]:
# =============================================================================
# SEL 5: PLOT TRAINING HISTORY
# =============================================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(history['lr'], label='LR', color='green')
axes[2].set_title('Learning Rate')
axes[2].set_yscale('log')
axes[2].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# =============================================================================
# SEL 6: CONFUSION MATRIX & REPORT
# =============================================================================
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model.load_state_dict(best_model_wts)
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# =============================================================================
# SEL 7: SIMPAN MODEL & CLASS NAMES
# =============================================================================
import json

model.load_state_dict(best_model_wts)
model.eval()

torch.save(model.state_dict(), 'model_herbalscan.pt')
print("Disimpan: model_herbalscan.pt")

try:
    example = torch.rand(1, 3, 224, 224).to(device)
    traced = torch.jit.trace(model, example)
    traced.save('model_herbalscan_jit.pt')
    print("Disimpan: model_herbalscan_jit.pt")
except Exception as e:
    print(f"JIT warning: {e}")

class_names_clean = [name.strip().lower().replace(" ", "_") for name in class_names]
with open('class_names.json', 'w') as f:
    json.dump(class_names_clean, f, indent=2)
print(f"Disimpan: class_names.json -> {class_names_clean}")

training_info = {
    "num_classes": num_classes,
    "class_names": class_names_clean,
    "best_val_accuracy": float(best_acc),
    "best_val_loss": float(best_loss),
    "epochs_trained": len(history['train_loss']),
    "total_epochs": NUM_EPOCHS,
    "model_architecture": "MobileNetV3-Large",
    "classifier_head": "960->256->128->N",
    "input_size": 224,
    "preprocessing": "ResizeWithPadding(224, white)",
    "dataset_size": len(full_dataset),
}
with open('training_info.json', 'w') as f:
    json.dump(training_info, f, indent=2)
print("Disimpan: training_info.json")

print("\nSelesai! Download model_herbalscan.pt dan class_names.json lalu masukkan ke folder models/ project kamu.")